## 🛒 SuperMarket Customer Satisfaction Prediction

Given *data about purchases made at three supermarkets*, let's try to predict the **satifaction level** of a given customer.

We will use a variety of regression models to make our predictions.

Data source: https://www.kaggle.com/datasets/faresashraf1001/supermarket-sales

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import LinearSVR, SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.base import BaseEstimator, TransformerMixin

import warnings
warnings.filterwarnings(action='ignore')

In [2]:
data = pd.read_csv('archive/SuperMarket Analysis.csv')
data

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,233-67-5758,Giza,Naypyitaw,Normal,Male,Health and beauty,40.35,1,2.0175,42.3675,1/29/2019,1:46:00 PM,Ewallet,40.35,4.761905,2.0175,6.2
996,303-96-2227,Cairo,Mandalay,Normal,Female,Home and lifestyle,97.38,10,48.6900,1022.4900,3/2/2019,5:16:00 PM,Ewallet,973.80,4.761905,48.6900,4.4
997,727-02-1313,Alex,Yangon,Member,Male,Food and beverages,31.84,1,1.5920,33.4320,2/9/2019,1:22:00 PM,Cash,31.84,4.761905,1.5920,7.7
998,347-56-2442,Alex,Yangon,Normal,Male,Home and lifestyle,65.82,1,3.2910,69.1110,2/22/2019,3:33:00 PM,Cash,65.82,4.761905,3.2910,4.1


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Sales                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  

### Preprocessing

In [4]:
df = data.copy()

In [5]:
df

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,233-67-5758,Giza,Naypyitaw,Normal,Male,Health and beauty,40.35,1,2.0175,42.3675,1/29/2019,1:46:00 PM,Ewallet,40.35,4.761905,2.0175,6.2
996,303-96-2227,Cairo,Mandalay,Normal,Female,Home and lifestyle,97.38,10,48.6900,1022.4900,3/2/2019,5:16:00 PM,Ewallet,973.80,4.761905,48.6900,4.4
997,727-02-1313,Alex,Yangon,Member,Male,Food and beverages,31.84,1,1.5920,33.4320,2/9/2019,1:22:00 PM,Cash,31.84,4.761905,1.5920,7.7
998,347-56-2442,Alex,Yangon,Normal,Male,Home and lifestyle,65.82,1,3.2910,69.1110,2/22/2019,3:33:00 PM,Cash,65.82,4.761905,3.2910,4.1


In [6]:
df = df.drop('Invoice ID', axis=1)

In [7]:
df

,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,Alex,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,1:08:00 PM,Ewallet,522.83,4.761905,26.1415,9.1
1,Giza,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29:00 AM,Cash,76.40,4.761905,3.8200,9.6
2,Alex,Yangon,Normal,Female,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,1:23:00 PM,Credit card,324.31,4.761905,16.2155,7.4
3,Alex,Yangon,Member,Female,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,8:33:00 PM,Ewallet,465.76,4.761905,23.2880,8.4
4,Alex,Yangon,Member,Female,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37:00 AM,Ewallet,604.17,4.761905,30.2085,5.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Giza,Naypyitaw,Normal,Male,Health and beauty,40.35,1,2.0175,42.3675,1/29/2019,1:46:00 PM,Ewallet,40.35,4.761905,2.0175,6.2
996,Cairo,Mandalay,Normal,Female,Home and lifestyle,97.38,10,48.6900,1022.4900,3/2/2019,5:16:00 PM,Ewallet,973.80,4.761905,48.6900,4.4
997,Alex,Yangon,Member,Male,Food and beverages,31.84,1,1.5920,33.4320,2/9/2019,1:22:00 PM,Cash,31.84,4.761905,1.5920,7.7
998,Alex,Yangon,Normal,Male,Home and lifestyle,65.82,1,3.2910,69.1110,2/22/2019,3:33:00 PM,Cash,65.82,4.761905,3.2910,4.1


In [8]:
# Split df into X and y
y = df['Rating'].copy()
X = df.drop('Rating', axis=1).copy()

In [9]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, shuffle=True, random_state=1)

In [10]:
X_train

,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Sales,Date,Time,Payment,cogs,gross margin percentage,gross income
731,Alex,Yangon,Normal,Male,Health and beauty,56.00,3,8.4000,176.4000,2/28/2019,7:33:00 PM,Ewallet,168.00,4.761905,8.4000
716,Alex,Yangon,Member,Female,Fashion accessories,71.46,7,25.0110,525.2310,3/28/2019,4:06:00 PM,Ewallet,500.22,4.761905,25.0110
640,Cairo,Mandalay,Member,Female,Food and beverages,98.79,3,14.8185,311.1885,2/23/2019,8:00:00 PM,Ewallet,296.37,4.761905,14.8185
804,Cairo,Mandalay,Member,Female,Electronic accessories,75.59,9,34.0155,714.3255,2/23/2019,11:12:00 AM,Cash,680.31,4.761905,34.0155
737,Giza,Naypyitaw,Normal,Male,Electronic accessories,58.76,10,29.3800,616.9800,1/29/2019,2:26:00 PM,Ewallet,587.60,4.761905,29.3800
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
767,Cairo,Mandalay,Normal,Male,Sports and travel,13.69,6,4.1070,86.2470,2/13/2019,1:59:00 PM,Cash,82.14,4.761905,4.1070
72,Cairo,Mandalay,Member,Female,Food and beverages,48.52,3,7.2780,152.8380,3/5/2019,6:17:00 PM,Ewallet,145.56,4.761905,7.2780
908,Alex,Yangon,Member,Female,Food and beverages,79.54,2,7.9540,167.0340,3/27/2019,4:30:00 PM,Ewallet,159.08,4.761905,7.9540
235,Alex,Yangon,Normal,Female,Sports and travel,93.14,2,9.3140,195.5940,1/20/2019,6:09:00 PM,Ewallet,186.28,4.761905,9.3140


In [11]:
y_train

731    4.8
716    4.5
640    6.4
804    8.0
737    9.0
      ... 
767    6.3
72     4.0
908    6.2
235    4.1
37     4.7
Name: Rating, Length: 700, dtype: float64

#### Constructing Pipeline

In [12]:
# Categorizing our features
{column: len(X_train[column].unique()) for column in X_train.select_dtypes('object').columns}

{'Branch': 3,
 'City': 3,
 'Customer type': 2,
 'Gender': 2,
 'Product line': 6,
 'Date': 89,
 'Time': 427,
 'Payment': 3}

In [13]:
binary_features = [
    'Customer type',
    'Gender'
]

date_features = [
    'Date'
]

time_features = [
    'Time'
]

nominal_features = [
    'Branch',
    'City',
    'Product line',
    'Payment'
]

In [14]:
# Create custom transformers for data and time features
class DateEncoder(TransformerMixin, BaseEstimator):
    def fit(self, X, y):
        self.is_fitted_=True
        return self
        
    def transform(self, X):
        X = X.copy()
        for column in X.columns:
            X[column] = pd.to_datetime(X[column])
            X[column + '_year'] = X[column].dt.year
            X[column + '_month'] = X[column].dt.month
            X[column + '_day'] = X[column].dt.day
            X = X.drop(column, axis=1)
        return X

In [15]:
date_encoder = DateEncoder()
date_encoder.transform(X_train[['Date']])

,Date_year,Date_month,Date_day
731,2019,2,28
716,2019,3,28
640,2019,2,23
804,2019,2,23
737,2019,1,29
...,...,...,...
767,2019,2,13
72,2019,3,5
908,2019,3,27
235,2019,1,20


In [16]:
# Create custom transformers for data and time features
class TimeEncoder(TransformerMixin, BaseEstimator):
    def fit(self, X, y):
        self.is_fitted_=True
        return self
        
    def transform(self, X):
        for column in X.columns:
            X = X.copy()
            X[column] = pd.to_datetime(X[column])
            X[column + '_hour'] = X[column].dt.hour
            X[column + '_minute'] = X[column].dt.minute
            X = X.drop(column, axis=1)
        return X

In [17]:
time_encoder = TimeEncoder()

time_encoder.transform(X_train[['Time']])

,Time_hour,Time_minute
731,19,33
716,16,6
640,20,0
804,11,12
737,14,26
...,...,...
767,13,59
72,18,17
908,16,30
235,18,9


In [18]:
# Construct Transformer Pipelines for each feature type
binary_transformer = Pipeline(steps=[
    ('ordinal', OrdinalEncoder())
])

date_transformer = Pipeline(steps=[
    ('date', DateEncoder())
])

time_transformer = Pipeline(steps=[
    ('time', TimeEncoder())
])

nominal_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder())
])

In [19]:
# Combine transformers with ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ('binary', binary_transformer, binary_features),
    ('date', date_transformer, date_features),
    ('time', time_transformer, time_features),
    ('nominal', nominal_transformer, nominal_features)
])

### Training

In [20]:
# Define models
models = {
    'Linear Regression                      ': LinearRegression(),
    'Linear Regression (L2 Regularization)  ': Ridge(),
    'Linear Regression (L1 Regularization)  ': Lasso(),
    'K-Nearest Neighbors                    ': KNeighborsRegressor(),
    'Neural Network                         ': MLPRegressor(),
    'Support Vector Machine (Linear Kernel) ': LinearSVR(),
    'Support Vector Machine (SBF Kernel   ) ': SVR(),
    'Decision Tree                          ': DecisionTreeRegressor(),
    'Random Forest                          ': RandomForestRegressor(),
    'Gradient Boosting                      ': GradientBoostingRegressor(),
    'XGBoost                                ': XGBRegressor(),
    'LightGBM                               ': LGBMRegressor(),
    'CatBoost                               ': CatBoostRegressor(verbose=0)
}

In [21]:
# Make a scaler
scaler = StandardScaler()

for name, model in models.items():
    # Construct the final pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', scaler),
        ('regressor', model)
    ])
    # Fit the pipeline
    pipeline.fit(X_train, y_train)
    print(name + " trained!")

Linear Regression                       trained!
Linear Regression (L2 Regularization)   trained!
Linear Regression (L1 Regularization)   trained!
K-Nearest Neighbors                     trained!
Neural Network                          trained!
Support Vector Machine (Linear Kernel ) trained!
Support Vector Machine (SBF Kernel    ) trained!
Decision Tree                           trained!
Random Forest                           trained!
Gradient Boosting                       trained!
XGBoost                                 trained!
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000254 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 160
[LightGBM] [Info] Number of data points in the train set: 700, number of used features: 21
[LightGBM] [Info] Start training from score 6.960143
[LightGBM] [Warning] No further splits with positive gain,

In [22]:
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](0,)",[]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](15,)","['Branch','City','Customer type',...,'cogs','gross margin percentage', 'gross income']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,15
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('binary', ...), ('date', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remai

### Results

In [23]:
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', scaler),
        ('regressor', model)
    ])
    print(name + "R^2 Score: {:.5f}".format(pipeline.score(X_test, y_test)))

Linear Regression                      R^2 Score: -0.01010
Linear Regression (L2 Regularization)  R^2 Score: -0.01007
Linear Regression (L1 Regularization)  R^2 Score: -0.00059
K-Nearest Neighbors                    R^2 Score: -0.13840
Neural Network                         R^2 Score: -0.06472
Support Vector Machine (Linear Kernel )R^2 Score: -0.03485
Support Vector Machine (SBF Kernel    )R^2 Score: -0.06062
Decision Tree                          R^2 Score: -0.96981
Random Forest                          R^2 Score: -0.03080
Gradient Boosting                      R^2 Score: -0.11482
XGBoost                                R^2 Score: -0.29028
LightGBM                               R^2 Score: -0.17491
CatBoost                               R^2 Score: -0.12378
